# Calculate ensemble statistics

This notebook demonstrates how to summarise an ensemble forecast with the `ensemble` sub-package of
earthkit-transforms. Each method reduces the data over the ensemble member dimension, so the result has the
same dimensions as the input apart from the members:

* `ekt.ensemble.mean` and `ekt.ensemble.std`, the ensemble mean and spread
* `ekt.ensemble.min`, `ekt.ensemble.max` and `ekt.ensemble.sum`
* `ekt.ensemble.reduce`, for any other reduction

We use the SEAS5 seasonal forecast of 2m temperature over Europe, with 25 members and three forecast
initialisations (January, February and March 2015) at 6-hourly resolution.

In [ ]:
from earthkit import data as ekd
from earthkit import plots as ekp
from earthkit import transforms as ekt
from earthkit.transforms._tools import earthkit_remote_test_data_file

# Get some demonstration SEAS5 data, this could be any url or path to an ensemble grib or netCDF file.
remote_seas5_file = earthkit_remote_test_data_file("seas5_2m_temperature_201501-201503_europe_1deg.grib")
seas5_data = ekd.from_source("url", remote_seas5_file)
ds = seas5_data.to_xarray(add_valid_time_coord=True)

ds

## Ensemble mean and spread

The ensemble dimension is detected automatically, from a `standard_name` of `"realization"` or from common
dimension names such as `number`, `member` or `realization`. The ensemble mean and standard deviation (spread)
are calculated with:

In [ ]:
ens_mean = ekt.ensemble.mean(ds)
ens_std = ekt.ensemble.std(ds)

ens_mean

## Ensemble range

The minimum and maximum over the members give the full range of the ensemble.

In [ ]:
ens_min = ekt.ensemble.min(ds)
ens_max = ekt.ensemble.max(ds)

ens_max

## Other reductions

`ekt.ensemble.reduce` accepts any reduction with the `how` argument: the name of an xarray reduction method,
an earthkit-transforms method or a numpy function, or a function called as `f(x, axis=axis)`. For example, the
ensemble median:

In [ ]:
ens_median = ekt.ensemble.reduce(ds, how="median")

ens_median

## Specify the ensemble dimension

If the ensemble dimension has a non-standard name and no `standard_name` attribute, it cannot be detected and
must be given with the `dim` argument. To demonstrate this, we rename the member dimension to `ensemble_id`
and drop its `standard_name` attribute.

In [ ]:
member_dim = ekt._tools.get_dim_key(ds, "realization")
ds_renamed = ds.rename({member_dim: "ensemble_id"})
ds_renamed["ensemble_id"].attrs.pop("standard_name", None)

ekt.ensemble.mean(ds_renamed, dim="ensemble_id")

## Plot the ensemble at a point

The statistics can be combined to show the forecast and its uncertainty, here for the forecast initialised on
2015-01-01 at a point in central England.

In [ ]:
point = {"latitude": 52, "longitude": -1, "forecast_reference_time": "2015-01-01"}

fig = ekp.Figure(rows=1, columns=1)
ts = fig.add_timeseries()
ts.fill_between(
    ens_min["2t"].sel(**point),
    ens_max["2t"].sel(**point),
    x="valid_time",
    units="celsius",
    label="Ensemble range (min to max)",
    color="#c6dbef",
    alpha=0.9,
)
ts.fill_between(
    (ens_mean - ens_std)["2t"].sel(**point),
    (ens_mean + ens_std)["2t"].sel(**point),
    x="valid_time",
    units="celsius",
    label="Ensemble mean ± 1 standard deviation",
    color="#6baed6",
    alpha=0.9,
)
ts.line(ens_mean["2t"].sel(**point), x="valid_time", units="celsius", label="Ensemble mean", color="#08306b")
ts.title("SEAS5 2m temperature at 52°N, 1°W, initialised 2015-01-01")
ts.legend()
ts.ylabel()

fig.show()